<h1>Light Spring - Full code</h1>

<h1> Libraries</h1>

In [ ]:
#Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# Motor Libraries
import pathlib, os, time
import libximc.highlevel as ximc
from libximc.highlevel._structure_types import home_settings_t

# Camera Libraries
#import ctypes
# import ft232
import instrumental
import pylablib as pll
pll.par["devices/dlls/thorlabs_tlcam"] = "C:/Program Files/Thorlabs/Scientific Imaging/ThorCam/"
from pylablib.devices import Thorlabs


from pathlib import Path
import os
import re

#Automated data scan
from src.SLM_lib_robert.Hologram import Hologram
from src.SLM_lib_robert.SLMDisplay import SLMScreen, screeninfo
from src.SLM_lib_robert.DataProcessing import *
from src.SLM_lib_robert.Visualizing import *
from imageio import imread
from time import sleep

#OGV
from scipy.optimize import minimize
from math import factorial

<h1>Motor and Camera Setup</h1>

In [ ]:
# ============================================================================================
# Set up the Motor
# Devices search
devices = ximc.enumerate_devices(ximc.EnumerateFlags.ENUMERATE_NETWORK | ximc.EnumerateFlags.ENUMERATE_PROBE)
if len(devices) == 0:
    print("The real devices were not found. A virtual device will be used.")
else:
    # Print real devices list
    print("Found {} real device(s):".format(len(devices)))
    for device in devices:
        print("  {}".format(device))
# Set up the correct one
device_uri = r"xi-com:\\.\COM7" # this is for devices connectd via USB (our case)
axis = ximc.Axis(device_uri)
axis.open_device() # To open the connection, you must manually call `open_device()` method

MIN_MOTOR_POS = -50000
MAX_MOTOR_POS = 50000
MOTOR_RANGE = 5000000 # nm

print( "Initial position:", axis.get_position().Position )
print( "Minimum position:", MIN_MOTOR_POS )
print( "Maximum position:", MAX_MOTOR_POS )
print( "Motor range:", MOTOR_RANGE, "nm" )
print( "Motor step:", MOTOR_RANGE / (MAX_MOTOR_POS-MIN_MOTOR_POS), "nm" )

def get_motor_pos():
    return axis.get_position().Position

def motor_to_origin():
    axis.command_move( 0, 0 )
    axis.command_wait_for_stop(100)

def motor_moveto( pos ):
    global MAX_MOTOR_POS
    global MIN_MOTOR_POS
    print(f"range: [{MIN_MOTOR_POS}; {MAX_MOTOR_POS}] current pos: {axis.get_position().Position}")
    if pos > MAX_MOTOR_POS:
        axis.command_move( MAX_MOTOR_POS, 0 )
        print( "Motor at maximum!" )
    elif pos < MIN_MOTOR_POS:
        axis.command_move( MIN_MOTOR_POS, 0 )
        print( "Motor at minimum!" )
    else:
        axis.command_move( pos, 0 )
    axis.command_wait_for_stop(100)

def motor_step( step ):
    pos = axis.get_position().Position
    if pos + step > MAX_MOTOR_POS:
        axis.command_move( MAX_MOTOR_POS, 0 )
        print( "Motor at maximum!" )
    elif pos + step < MIN_MOTOR_POS:
        axis.command_move( MIN_MOTOR_POS, 0 )
        print( "Motor at minimum!" )
    else:
        axis.command_move( pos + step, 0 )
    axis.command_wait_for_stop(100)

def motor_home(reset_max_pos:bool = True):
    """
    Homes motor by checking for right-sided stop button and setting the position to 0.
    The driving direction is negative afterwards!

    :param reset_max_pos: reset motor max positions to -150000, 0
    :type reset_max_pos: bool
    """
    home_settings = home_settings_t(
        FastHome=10000,
        uFastHome=0,
        SlowHome=2000,
        uSlowHome=0,
        HomeDelta=-5,
        uHomeDelta=0,
        #maybe still adjustable for better settings?
        HomeFlags=ximc.HomeFlags.HOME_DIR_FIRST|ximc.HomeFlags.HOME_STOP_FIRST_LIM
    )
    global MIN_MOTOR_POS
    global MAX_MOTOR_POS
    print("Motor is homing...")
    axis.set_home_settings(home_settings)
    axis.command_home()
    axis.command_wait_for_stop(100)
    axis.command_zero()
    if reset_max_pos:
        MAX_MOTOR_POS=0
        MIN_MOTOR_POS=-150000
    print("Finished homing")

# Set up the Camera
cameras_SN_list = Thorlabs.list_cameras_tlcam()
print("Camera list = " + str(cameras_SN_list)) 
cam_selected = 17943
print( "Selected Camera: ", cam_selected )
try: 
    ccd.close()
except: 
    pass
ccd = Thorlabs.ThorlabsTLCamera( cam_selected )
# Set up the exposure
EXPOSURE = 1
ccd.set_exposure( EXPOSURE )
print( "Exposure set to ", EXPOSURE )
PIXEL_SIZE = 3.3 # micron ?

######################################################################################################
# Funtions for image processing
def take_picture(exposure:float, remove_dead_pixels = False):
    ccd.set_exposure(exposure)
    img = np.asarray(ccd.snap())

    if remove_dead_pixels:
        dead_pixels = img / img.max() > 0.95
        img[ dead_pixels ] = 0
    return img

def crop_defined(image:np.ndarray,center:tuple[int,int], abs_x:int, abs_y:int)->np.ndarray:
    """ 
    Crops the given image to a rectangular geometrz defined by the following parameters:

    :param image: Image data to be cropped
    :type image: numpy.ndarray

    :param center: Center coordinate of the rectangle
    :type center: tuple[int,int]

    :param abs_x: absolute width in x direction (horizontal)
    :type abs_x: int

    :param abs_y: absolute hight in y direction (vertically)
    :type abs_y: int
    """
    min_x = center[0] - int(abs_x/2)
    max_x = center[0] + int(abs_x/2)
    min_y = center[1] - int(abs_y/2)
    max_y = center[1] + int(abs_y/2)
    if min_x < 0:
        min_x = 0
    if max_x > image.shape[0]:
        max_x = image.shape[0]
    if min_y < 0:
        min_y = 0
    if max_y > image.shape[1]:
        max_y = image.shape[1]
    newImage = image[min_x:max_x, min_y:max_y]
    return newImage

def analyseFFT(image:np.ndarray, kmin:float, plot:bool = True): #Add to the lib
    nx, ny = image.shape
    kx = np.fft.fftshift( np.fft.fftfreq( nx, PIXEL_SIZE ) ) * 2*np.pi
    ky = np.fft.fftshift( np.fft.fftfreq( ny, PIXEL_SIZE ) ) * 2*np.pi
    KX, KY = np.meshgrid( ky, kx )
    # fourier transform
    fft = np.fft.fftshift(np.fft.fft2(image))
    fft /= np.max( np.abs( fft ) )
    # get intensity maximum of higher frequency components
    intensity = np.max( np.abs( fft[ np.sqrt(KX**2+KY**2) > kmin ] ) )
    
    if plot:
        plt.close("all")
        #print('plotting')
        fft[ np.sqrt(KX**2+KY**2) < kmin ] = 0
        
        fig, [ax1, ax2] = plt.subplots( 1, 2, figsize=[10,5] )
        
        plt.sca(ax1)
        plt.imshow( image,
                    origin="lower",
                    cmap="rainbow" )
        plt.colorbar()
        #plt.xlim( 1000, 2000 )
        #plt.ylim( 2500, 3500 )
        plt.title( "CCD" )
        
        plt.sca(ax2)
        plt.imshow( np.abs( fft ),
                    extent=[ np.min(kx), np.max(kx), np.min(ky), np.max(ky) ],
                    origin="lower",
                    cmap="hot",
                    norm=mpl.colors.LogNorm( vmin=1e-5 ))
        c1 = plt.Circle((0,0),kmin,color = "green")
        ax2.add_patch(c1)
        plt.colorbar()
        plt.title( "Absolute Value of FFT" )
        
        plt.suptitle( "Pos = "+ str( get_motor_pos() ) )
        plt.show()

    return( get_motor_pos(),intensity )

def get_object_position( obj, plot=False, rmv_dead_pixels=False ):

    if rmv_dead_pixels:
        dead_pixels = obj / obj.max() > 0.95
        obj[ dead_pixels ] = 0

    nx, ny = obj.shape
    obj_x = int( np.average( np.arange(nx), weights=np.sum(obj.T, 0) ) )
    obj_y = int( np.average( np.arange(ny), weights=np.sum(obj.T, 1) ) )
    obj_n = np.min( [ obj_x, obj_y, nx-obj_x, ny-obj_y ] )

    if plot:
        fig, ax = plt.subplots( 1, 2, figsize=[10,5] )

        ax[0].imshow( obj.T, cmap="rainbow", origin="lower", aspect="auto" )
        ax[0].scatter( [obj_x], [obj_y], c="black", marker="x" )
        ax[0].plot( np.sum(obj.T, 0)/np.max(np.sum(obj.T, 0)) * ny/4, c="k", lw=1 )
        ax[0].plot( np.sum(obj.T, 1)/np.max(np.sum(obj.T, 1)) * nx/4, np.arange(ny), c="k", lw=1 )

        ax[0].axvline( obj_x, c="r", lw=1, ls="--" )
        ax[0].axhline( obj_y, c="r", lw=1, ls="--" )

        ax[1].imshow( obj[obj_x-obj_n:obj_x+obj_n, obj_y-obj_n:obj_y+obj_n].T, cmap="rainbow", origin="lower" )
        ax[1].axvline( obj_n, c="black", ls="--" )
        ax[1].axhline( obj_n, c="black", ls="--" )
        plt.show()

    return obj_x, obj_y, obj_n

<h1> SLM Hologram </h1>

In [ ]:
slm = SLMScreen(1)
h = Hologram()
h.loadPhaseCalibrationFile(Path(r"C:\Users\Voxel\Desktop\Data_analysis_LS\Correction_patterns/CAL_LSH0803840_785nm.bmp"),
                      plot=False)
h.loadVoltageCalibrationFile(Path(r"C:\Users\Voxel\Desktop\LightSprings\polyfit_par.txt"))
h.setZernikeCorrection(verAstic=.03)
h.lightspring(30,1,9,5.63,6.05)
h.plotHolo()
h.plotModes()
slm.updateImage(h.convertHolo())
# for i in range(20,100,20):
#     h.lightspring(i,1,9,5.63,6.05)
#     slm.updateImage(h.convertHolo())
#     h.plotHolo()
#     sleep(2)
# slm.close()

<h2>Close Hologram </h2>

In [ ]:
#close the slm Window without crashing the kernel
slm.close()

<h1> Temporal Overlap - FFT </h1>

<h2>Single Picture</h2>

In [ ]:
# single picture
%matplotlib tk
#motor_home()
#-115700, -112800,
motor_moveto(-112800)
save = 1
savePath = pathlib.Path(r"C:\Users\Voxel\Desktop\LightSprings\Data")/ pathlib.Path(time.strftime("%y%m%d",time.gmtime())) /"correctionData"
if not os.path.exists(savePath):
    os.makedirs(savePath)
angle = 1
beams = ["Ref", "Main","Interf"]
beam = 2
pos = get_motor_pos()
filename = f"LightSpring_Filter_7OAM_{angle}_{beams[beam]}"
filename = f"Lightspring_80_86OAM_centered_{beams[beam]}_2"
filename = f"Vortex_70OAM_pos{pos}_{beams[beam]}"
filename = f"correctionWave_{beams[beam]}_{pos}"

fig = plt.figure()
#while True:
img = take_picture(0.001, 1)
#img = np.load(r"C:\Users\Voxel\Desktop\LightSprings\Data\19-03-25\OAM-Anglesfilter\0_deg.npy")
#print(img.shape)
#print(img)
cropped = img#crop_defined(img,(500,700), 650, 650)# with grating: crop_defined(img,(250,600), 450, 450)
#print(cropped.shape)
#print(cropped)
ax = plt.gca()
ax.imshow(cropped, cmap="rainbow")
analyseFFT(cropped, kmin=0.0)
if save:
    np.savetxt((savePath/filename).with_suffix(".csv"), img, delimiter=";")
    np.save(savePath/filename, img)

<h2> Full Temporal Overlap Scan</h2>

In [ ]:
#setup
%matplotlib inline
remove_dead_pixels= 1
ccd.set_exposure(0.001)
pos_min, pos_max, step = -150000, -1000, 1500
kmin = 0.06
    # crop
do_crop = 0
c_center, c_x, c_y = (600,1150), 350, 350

plot = True
save = False
OAM_charge = 10
filenames = "C:/Users/Voxel/Desktop/LightSprings/Data/04-07-24/LS_10_8_8_6_6_4_4_2_2_0_0/"
####
data_pos = []
data_int = []
motor_home()
motor_moveto( pos_min )
ccd.start_acquisition( nframes=1 )
while get_motor_pos() < pos_max:  
    # get data from camera
    ccd.wait_for_frame()
    img = ccd.read_oldest_image()
    if remove_dead_pixels:
        dead_pixels = img / img.max() > 0.95
        img[ dead_pixels ] = 0
    # fourier transform
    if do_crop:
        cropped = crop_defined(img, c_center, c_x, c_y)
    else:
        cropped = img

    res = analyseFFT(cropped, kmin, plot)
    
    data_pos.append(res[0])
    data_int.append(res[1])

    # save
    if save:
        img_save = crop_img( img )
        if not os.path.exists(filenames+ dataset):
            os.mkdir(filenames+ dataset)
        np.save( filenames+ dataset +"motor%06i"%( get_motor_pos() - MIN_MOTOR_POS ), img_save )

    print( data_pos[-1], data_int[-1], "       ", end="\r" )
    
    motor_step(step)
ccd.stop_acquisition()

<h2> Plot Temporal Overlap Scan </h2>

In [ ]:
xmin, xmax = pos_min, pos_max
%matplotlib tk
fig = plt.figure()
plt.scatter( data_pos, data_int, s=1 )
plt.axvline(xmin, lw=1, c="r", ls="--")
plt.axvline(xmax, lw=1, c="r", ls="--")
#plt.xlim( -11000, -4000 )
plt.xlabel( "Motor Position" )
plt.ylabel( "Maximum off axis FFT component" )
plt.show()

fig = plt.figure()
plt.scatter( data_pos, data_int, s=1 )
plt.xlim( xmin, xmax )
plt.xlabel( "Motor Position" )
plt.ylabel( "Maximum off axis FFT component" )
plt.grid()
plt.show()


<h1>Data Acquisition</h1>

<h2> Automated scan to aquire data </h2>

In [ ]:
%matplotlib inline
# aquisition
show = 0
remove_dead_pixels= 1
ccd.set_exposure(0.001)
pos_min, pos_max, step = -115700, -112800, 60
# path and names
path = pathlib.Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\\"+time.strftime("%y%m%d",time.gmtime()))
structureName = "lighspring"
topologicalCharge = "30"
additionalName = "sligthBiggerAngle"
subfolder = pathlib.Path(f"{structureName}_OAM{topologicalCharge}_{additionalName}")
#
####
stem = structureName+f"_tc{topologicalCharge}"+f"_{additionalName}"
data_pos = []
data_int = []
motor_home()
motor_moveto( pos_min )
ccd.start_acquisition( nframes=1 )
n_images = np.abs(pos_max - pos_min) // step
print(f"Starting to aquire images:\n  savepath = {path}\n  stemname = {stem}\n  n_images = {n_images}")
i = 1
if not subfolder.__str__ == "":
    path = path/subfolder
while (get_motor_pos() < pos_max):  
    # get data from camera
    ccd.wait_for_frame()
    img = ccd.read_oldest_image()
    if remove_dead_pixels:
        dead_pixels = img / img.max() > 0.95
        img[ dead_pixels ] = 0
    # # fourier transform
    # if do_crop:
    #     cropped = crop_defined(img, c_center, c_x, c_y)
    # else:
    #     cropped = img

    # res = analyseFFT(cropped, kmin, plot)
    # data_pos.append(res[0])
    # data_int.append(res[1])
    if show:
        fig = plt.figure()
        a = plt.imshow(img, cmap = "inferno")
        plt.colorbar(a)
        plt.show()
    # save
    if not os.path.exists(path):
        os.makedirs(path)
    np.save(path/pathlib.Path(stem+"_motorPos%i"%( get_motor_pos())),
            img)
    print(f"aquired image {i}/{n_images}")
    i = i +1
    motor_step(step)
ccd.stop_acquisition()
print("Finished")

<h2>Multi Parameter Measurement sets </h2>

In [ ]:
%matplotlib inline
holo = Hologram()

SCREENID = 1
SLM = SLMScreen(SCREENID)
holo.loadPhaseCalibrationFile(
    Path(r"C:\Users\Voxel\Desktop\Data_analysis_LS\Correction_patterns/CAL_LSH0803840_785nm.bmp")
)
holo.loadVoltageCalibrationFile(
    Path(r"C:\Users\Voxel\Desktop\LightSprings\polyfit_par.txt")
)
holo.setZernikeCorrection(verAstic=0.03)
#keys
N_FREQUENCYBINS = "n"
TOPOLOGICALCHARGE = "OAM"
SLM_POSITION = "SLM"
DELTA_L = "dl"
TOTAL_DL = "totaldl"

parameterSet = {
    SLM_POSITION:[9],
    N_FREQUENCYBINS:np.linspace(3,19,17, dtype=int),
    TOPOLOGICALCHARGE:[25, 30, 35, 40, 45, 50, 55, 60],
    DELTA_L:[1],
    TOTAL_DL:None #np.linspace(6,60,10, dtype=int)
}

def buildHologram(parameters:dict, rMin, rMax, holo:Hologram = None, plot = False):
    if holo is None: holo = Hologram()
    holo.resetHolo()
    # holo.lightspring(
    #     OAM=parameters[TOPOLOGICALCHARGE],
    #     deltaL=parameters[DELTA_L],
    #     nSample=parameters[N_FREQUENCYBINS],
    #     rMin=rMin, rMax=rMax
    # )
    # centered OAM
    OAMs = np.ones((parameters[N_FREQUENCYBINS]))*parameters[TOPOLOGICALCHARGE]
    if parameters[N_FREQUENCYBINS] % 2 == 0:
        OAMs = np.linspace(parameters[TOPOLOGICALCHARGE]-np.floor(parameters[N_FREQUENCYBINS]/2),
                                parameters[TOPOLOGICALCHARGE]+np.floor(parameters[N_FREQUENCYBINS]/2)-1,parameters[N_FREQUENCYBINS])
    else:
        OAMs = np.linspace(parameters[TOPOLOGICALCHARGE]-np.floor(parameters[N_FREQUENCYBINS]/2),
                                parameters[TOPOLOGICALCHARGE]+np.floor(parameters[N_FREQUENCYBINS]/2),parameters[N_FREQUENCYBINS])
    radia = holo._linearInterpolRadius(parameters[N_FREQUENCYBINS],10)[0]
    holo.customFunction(radia, OAMs)
    print(f"OAMs = {OAMs}")
    if plot:
        holo.plotHolo()
        holo.plotModes()
    
    # ## centered OAM with fixed total DeltaL
    # OAMs = np.ones((parameters[N_FREQUENCYBINS]))*parameters[TOPOLOGICALCHARGE]
    # dl = parameters[TOTAL_DL] / parameters[N_FREQUENCYBINS]
    # if parameters[N_FREQUENCYBINS] % 2 == 0:
    #     OAMs = OAMs + np.round(np.linspace(-np.floor(parameters[N_FREQUENCYBINS]/2),
    #                         np.floor(parameters[N_FREQUENCYBINS]/2)-1,
    #                         parameters[N_FREQUENCYBINS])*dl,0)
    # else:
    #     OAMs = OAMs + np.round(np.linspace(-np.floor(parameters[N_FREQUENCYBINS]/2),
    #                         np.floor(parameters[N_FREQUENCYBINS]/2),
    #                         parameters[N_FREQUENCYBINS])*dl,0)
        
    # radia = holo._linearInterpolRadius(parameters[N_FREQUENCYBINS],10)[0]
    # holo.customFunction(radia, OAMs)
    # print(f"OAMs = {OAMs}")
    # if plot:
    #     holo.plotHolo()
    #     holo.plotModes()

def scanForSLMPos(slmPos, structureName, rMin, rMax,
                  pos_min, pos_max, step):
    holo.rMin, holo.rMax = rMin, rMax
    for n in parameterSet[N_FREQUENCYBINS]:
        for OAM in parameterSet[TOPOLOGICALCHARGE]:
            if not parameterSet[DELTA_L] is None:
                for dl in parameterSet[DELTA_L]:
                    # buildHologram({SLM_POSITION:slmPos,
                    #      N_FREQUENCYBINS:n,
                    #      TOPOLOGICALCHARGE:OAM,
                    #      DELTA_L:dl},rMin,rMax,holo)
                    # sleep(2)
                    scanSingle(
                        {SLM_POSITION:slmPos,
                        N_FREQUENCYBINS:n,
                        TOPOLOGICALCHARGE:OAM,
                        DELTA_L:dl},
                        structureName, rMin,rMax,
                        pos_min, pos_max, step
                    )
            elif not parameterSet[TOTAL_DL] is None:
                for tdl in parameterSet[TOTAL_DL]:
                    scanSingle(
                        {SLM_POSITION:slmPos,
                        N_FREQUENCYBINS:n,
                        TOPOLOGICALCHARGE:OAM,
                        TOTAL_DL:tdl},
                        structureName, rMin,rMax,
                        pos_min, pos_max, step
                    )

def scanSingle(parameters:dict, structureName, rMin,rMax,
               pos_min, pos_max, step,
               show = 0, remove_dead_pixels = 1):
    #build hologram
    buildHologram(parameters,rMin,rMax,holo, True)
    SLM.updateImage(holo.convertHolo())
    # aquisition
    ccd.set_exposure(0.001)
    # path and names from sructure
    path = pathlib.Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\\"+time.strftime("%y%m%d",time.gmtime()))
    structureName = structureName
    topologicalCharge = parameters[TOPOLOGICALCHARGE]
    additionalName = "_".join([f"{key}={val}" for key, val in parameters.items() if not (key == TOPOLOGICALCHARGE)])
    subfolder = pathlib.Path(f"{structureName}_OAM{topologicalCharge}_{additionalName}")
    #
    ####
    stem = structureName+f"_tc{topologicalCharge}"+f"_{additionalName}"
    motor_home()
    motor_moveto( pos_min )
    ccd.start_acquisition( nframes=1 )
    n_images = np.abs(pos_max - pos_min) // step
    print(f"Starting to aquire images:\nParameters = {parameters}\n  savepath = {path}\n  stemname = {stem}\n  n_images = {n_images}")
    i = 1
    if not subfolder.__str__ == "":
        path = path/subfolder
    while (get_motor_pos() < pos_max):  
        # get data from camera
        ccd.wait_for_frame()
        img = ccd.read_oldest_image()
        if remove_dead_pixels:
            dead_pixels = img / img.max() > 0.95
            img[ dead_pixels ] = 0

        if show:
            fig = plt.figure()
            a = plt.imshow(img, cmap = "inferno")
            plt.colorbar(a)
            plt.show()
        # save
        if not os.path.exists(path):
            os.makedirs(path)
        np.save(path/pathlib.Path(stem+"_motorPos%i"%( get_motor_pos())),
                img)
        print(f"aquired image {i}/{n_images}")
        i = i +1
        motor_step(step)
    ccd.stop_acquisition()
    print("Finished Scan")

scanForSLMPos(parameterSet[SLM_POSITION][0], "lightspring", 5.63, 6.05, -58000,-53900,100)

<h1>Data Processing</h1>

<h2>Single Reconstruction</h2>

In [ ]:
%matplotlib tk
rotangle = 2.5
timing = 6201
cx=437
cy=359
rin=33
rout=45
crop_pos = crop_pos = ((550,660),700)
path = Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\250403\surfaceplot")
file_base = Path("LightSpring_Filter_7OAM__Interf.npy")
file = "Vortex_70OAM_pos10_Interf.npy"
reconstruct_data( path/file, cx, cy, rin, rout, crop_pos, plot_all=True, plot_reconstruction=True,fft_crop_radius=6,circle_shift=(0,0) )

<h2>Reconstruction from Folders</h2>

In [ ]:
%matplotlib inline
# 30 till 90 OAM
# cx=436
# cy=359
# rin=33 
# rout=44
# crop_pos = crop_pos = ((550,660),1000)
# shift = (np.linspace(0,3,28, dtype=int), np.zeros(28, dtype=int))
root = Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\250513")
path= root/Path("lightSpring_OAM30_smallAngle_n=9")
#fileName = r"C:\Users\Voxel\Desktop\LightSprings\Data\250513" #Single File
ellipse = (
    500,    #x
    500,    #y
    90,     #A
    90,     #B
    10,     #thickness
    0       #angle
)
crop_pos = ((600,500),1000)
shift = (np.around(np.linspace(0,5,200,0)), np.zeros(156, dtype=int))

print(shift)
reconstructFromFolder(path, *ellipse, crop_pos, removeWrongExposure=True, phaselock=0, shift=shift,
                      plot_exp_hist=True, plot_all = True,
                      fft_crop_radius=5.5)#, subtract_ref=root/Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\250512\lighspring_OAM50_n=9_sligthBiggerAngle\correctionData\correctionWave_Interf_-112800.npy"))
# Tests phase tilt
#reconstructFromFolder(path, cx, cy, rin, rout, crop_pos, tilt_correction=1, phaselock=0, plot_all=0, shift=shift)

<h2>Manual FFT Reconstruction</h2>

In [ ]:
# from nfft import nfft
# from BPF_calib import lambda_f
# lambdas = lambda_f(rotangles)
# print(lambdas)
# FIELD = np.fft.fft(hypervolume)
%matplotlib inline
# FIELD = np.fft.fft(phaselockField2(hypervolume))
print(omegas)
FIELD = manualFFT(hypervolume, omegas, 2e-15)
print(FIELD)
plt.figure()
plt.imshow(np.abs(FIELD),cmap="inferno",vmin=0)
plt.colorbar()
plt.title("reconstructed field")
print(FIELD.shape)

<h2>Animation</h2>

In [ ]:
%matplotlib tk
# animation
from matplotlib.animation import FuncAnimation, PillowWriter

fig, ax = plt.subplots()
i = 0,
frames = 200
t = np.linspace(-200e-15,200e-15,frames)
ln = ax.imshow(np.abs(manualFFT(hypervolume, omegas, t[0])), cmap = "inferno", vmin = 0)
cbar = fig.colorbar(ln, cmap = "inferno")
def init():
    print(i)
    return [ln]

def update(frame):
    ax.set_title(f"[{t[0]*1e15}; {t[-1]*1e15}]fs; {frames} frames;   t = {round(t[frame]*1e15,2)}fs")
    ln.set_data(np.abs(manualFFT(hypervolume, omegas, t[frame], 1)))
    cbar.update_normal(ln)
    return [ln]

ani = FuncAnimation(fig, update, frames = frames, blit= True)
# To save the animation using Pillow as a gif
writer = PillowWriter(fps=15,
           metadata=dict(artist='Me'),
           bitrate=1800)
ani.save(Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\31-03-25")/"lightspring_phaselock_subtraction.gif", writer=writer)

<h1>OGV Analysis</h1>

In [ ]:
path_root = Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\250506\lightspring_OAM25_SLM=9_n=7_dl=1")
fname = Path("lightspring_tc25_SLM=9_n=7_dl=1_motorPos-55881.npy")
#paths = [r"C:\Users\Voxel\Desktop\LightSprings\Data\250411\OAM50to56dl1Check\Lightspring_tc50to56_dl1Check_motorPos-62077.npy",
 #       r"C:\Users\Voxel\Desktop\LightSprings\Data\250411\OAM50to56dl1Check\Lightspring_tc50to56_dl1Check_motorPos-62584.npy",
  #      r"C:\Users\Voxel\Desktop\LightSprings\Data\250411\OAM80to86dl1Check\Lightspring_tc80to86_dl1Check_motorPos-61730.npy",
   #     r"C:\Users\Voxel\Desktop\LightSprings\Data\250411\OAM80to86dl1Check\Lightspring_tc80to86_dl1Check_motorPos-62469.npy",
    #    r"C:\Users\Voxel\Desktop\LightSprings\Data\250411\OAM120to126dl1Check\Lightspring_tc120to126_dl1Check_motorPos-61864.npy",
     #   r"C:\Users\Voxel\Desktop\LightSprings\Data\250411\OAM120to126dl1Check\Lightspring_tc120to126_dl1Check_motorPos-62235.npy",
      #  r"C:\Users\Voxel\Desktop\LightSprings\Data\250417\lightspring_OAM18_dl1\lightspring_tc18_dl1_motorPos-62481.npy",
       # r"C:\Users\Voxel\Desktop\LightSprings\Data\250417\lightspring_OAM18_dl1\lightspring_tc18_dl1_motorPos-62787.npy",
       # r"C:\Users\Voxel\Desktop\LightSprings\Data\250417\lightspring_OAM25_dl-1\lightspring_tc25_dl-1_motorPos-62183.npy",
       # r"C:\Users\Voxel\Desktop\LightSprings\Data\250417\lightspring_OAM25_dl-1\lightspring_tc25_dl-1_motorPos-62721.npy",
       # r"C:\Users\Voxel\Desktop\LightSprings\Data\250417\lightspring_OAM30_dl1\lightspring_tc30_dl1_motorPos-62129.npy",
       # r"C:\Users\Voxel\Desktop\LightSprings\Data\250417\lightspring_OAM30_dl1\lightspring_tc30_dl1_motorPos-62619.npy"
#]
#for file in paths:
img = np.load(Path(path_root/fname))
plt.figure()
plt.imshow(img)
plt.show()
r = findRadius(img,(686,524),15,.6,sigma_Fit=0.5,showStatistics=True)
r_m = calculateRadius(r)*1e-3
OGV = calculateOGV(r_m)
print(f"LOGV calculated = {OGV/300000} /c")
print(f"{r_m*1e6} um")
